# 08c: Debug Notebook - 2D Array Operations (L3)

## Purpose
Validate DSL 2D array operations at Level 3 (RVec<RVec> / nested arrays).

## Phase 13.6.G Debug Notebook

This notebook tests:
- 2D column slicing: `cluster_Q[:, 0]` (first cluster per track)
- 2D row slicing: `cluster_Q[0, :]` (all clusters of first track)
- 2D combined slicing: `cluster_Q[:2, :2]`
- 2D reductions: `Sum(cluster_Q)` (nested sum)

## Setup

In [ ]:
# Path setup - ensure RDataFrameDSL is importable
import sys
import os

# Add parent directory to path if running from examples/
notebook_dir = os.path.dirname(os.path.abspath('.'))
if 'RDataFrameDSL' not in sys.modules:
    for path in ['.', '..', notebook_dir]:
        if os.path.exists(os.path.join(path, 'RDataFrameDSL')):
            sys.path.insert(0, os.path.abspath(path))
            break

In [ ]:
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import time

from RDataFrameDSL import DSLCompiler
from RDataFrameDSL.verbosity import VERBOSE_DEFAULT, VERBOSE_FULL

logging.basicConfig(level=logging.INFO)
plt.rcParams['figure.figsize'] = (6, 4)

print("Setup complete")

## Generate Test Data

In [ ]:
from tests.generators.toy_nd import generate_nd_2d_root

filename = generate_nd_2d_root(size='S', seed=42)
rdf = ROOT.RDataFrame("Events", filename)

print("Available columns:", list(rdf.GetColumnNames()))
print(f"Number of events: {rdf.Count().GetValue()}")

# Schema for 2D operations
schema = {
    'event_id': 'long',
    'n_tracks': 'int',
    'event_weight': 'double',
    'track_pt': 'RVec<double>',
    'track_eta': 'RVec<double>',
    'cluster_Q': 'RVec<RVec<double>>',
    'cluster_x': 'RVec<RVec<double>>',
    'cluster_y': 'RVec<RVec<double>>',
}

## Section 1: 2D Column Slicing

### Description
Extract specific cluster indices from all tracks.

### What we're testing
- First cluster: `cluster_Q[:, 0]` → RVec of first cluster per track
- Last cluster: `cluster_Q[:, -1]` → RVec of last cluster per track

### If it fails
Check 2D slice handling in `_visit_Subscript()` and `backend_cpp.py` nested loop generation.

In [ ]:
dsl = DSLCompiler(schema)

# 2D column slicing
dsl.define("first_cluster_Q", "cluster_Q[:, 0]")
dsl.define("last_cluster_Q", "cluster_Q[:, -1]")  # Phase 13.6.G: 2D negative index

print(dsl.describe_structure())

In [ ]:
# Get raw data
raw = rdf.Range(5).AsNumpy(['cluster_Q'])

# Execute DSL
applied = dsl.apply(rdf)
result = applied.Range(5).AsNumpy(['cluster_Q', 'first_cluster_Q', 'last_cluster_Q'])

# Validate
for i in range(5):
    Q = raw['cluster_Q'][i]  # array of arrays (RVec<RVec>)
    
    # First cluster per track - convert each track to numpy
    expected_first = np.array([np.array(track)[0] if len(track) > 0 else 0 for track in Q])
    actual_first = np.array(result['first_cluster_Q'][i])
    assert np.allclose(expected_first, actual_first), f"Event {i}: first_cluster_Q mismatch"
    
    # Last cluster per track (Phase 13.6.G: 2D negative index)
    # ROOT RVec doesn't support negative indexing in Python, use len-1
    expected_last = np.array([np.array(track)[len(track)-1] if len(track) > 0 else 0 for track in Q])
    actual_last = np.array(result['last_cluster_Q'][i])
    assert np.allclose(expected_last, actual_last), f"Event {i}: last_cluster_Q mismatch"

print("✓ 2D column slicing: All validations passed")

## Section 2: 2D Row Slicing

### Description
Extract all clusters from specific tracks.

### What we're testing
- First track clusters: `cluster_Q[0, :]`
- Last track clusters: `cluster_Q[-1, :]`

In [ ]:
dsl2 = DSLCompiler(schema)

dsl2.define("track0_clusters", "cluster_Q[0, :]")
dsl2.define("last_track_clusters", "cluster_Q[-1, :]")  # Phase 13.6.G: 2D negative index

applied2 = dsl2.apply(rdf)
result2 = applied2.Range(5).AsNumpy(['cluster_Q', 'track0_clusters', 'last_track_clusters'])

# Validate
for i in range(5):
    Q = raw['cluster_Q'][i]
    n_tracks = len(Q)
    
    if n_tracks > 0:
        # First track
        expected_t0 = np.array(Q[0])  # Convert RVec to numpy
        actual_t0 = np.array(result2['track0_clusters'][i])
        assert np.allclose(expected_t0, actual_t0), f"Event {i}: track0 mismatch"
        
        # Last track (Phase 13.6.G: 2D negative index)
        # ROOT RVec doesn't support negative indexing in Python, use n_tracks-1
        expected_last = np.array(Q[n_tracks - 1])
        actual_last = np.array(result2['last_track_clusters'][i])
        assert np.allclose(expected_last, actual_last), f"Event {i}: last track mismatch"

print("✓ 2D row slicing: All validations passed")

## Section 3: 2D Combined Slicing

### Description
Extract subsets using both dimensions.

### What we're testing
- Top-left: `cluster_Q[:2, :2]` (first 2 tracks, first 2 clusters)
- Bottom-right: `cluster_Q[-2:, -2:]`

In [ ]:
dsl3 = DSLCompiler(schema)

dsl3.define("top_left", "cluster_Q[:2, :2]")

applied3 = dsl3.apply(rdf)
result3 = applied3.Range(5).AsNumpy(['cluster_Q', 'top_left'])

# Validate
for i in range(5):
    Q = raw['cluster_Q'][i]
    
    # Expected: first 2 tracks, first 2 clusters each
    # Must convert RVec to numpy for slicing
    expected = []
    for j, t in enumerate(Q):
        if j >= 2:
            break
        t_np = np.array(t)
        expected.append(t_np[:2] if len(t_np) >= 2 else t_np)
    
    actual = result3['top_left'][i]
    
    # Compare nested structure
    assert len(actual) == len(expected), f"Event {i}: track count mismatch"
    for j in range(len(expected)):
        assert np.allclose(expected[j], np.array(actual[j])), f"Event {i}, Track {j}: cluster mismatch"

print("✓ 2D combined slicing: All validations passed")

## Section 4: 2D Reductions

### Description
Test nested reductions on 2D arrays.

### What we're testing
- `Sum(cluster_Q)` → sum of all clusters across all tracks

In [ ]:
dsl4 = DSLCompiler(schema)

dsl4.define("total_Q", "Sum(cluster_Q)")

df4 = dsl4.to_pandas(rdf, ['total_Q'])

# Validate
raw_all = rdf.AsNumpy(['cluster_Q'])
expected_total = []
for event_Q in raw_all['cluster_Q']:
    total = sum(sum(track) for track in event_Q)
    expected_total.append(total)

assert np.allclose(df4['total_Q'], expected_total), "Total Q mismatch"

print("✓ 2D reductions: All validations passed")
df4.head()

## CPU Benchmarking

In [ ]:
def benchmark(name, func, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        result = func()
        times.append(time.perf_counter() - start)
    print(f"{name}: {np.mean(times):.4f}s ± {np.std(times):.4f}s")
    return result, times

# DSL 2D sum
dsl_bench = DSLCompiler(schema)
dsl_bench.define("total_Q", "Sum(cluster_Q)")
_, times_dsl = benchmark("DSL Sum(cluster_Q)", 
                         lambda: dsl_bench.to_pandas(rdf, ['total_Q']))

# Python nested loop
def python_nested_sum():
    data = rdf.AsNumpy(['cluster_Q'])
    return [sum(sum(t) for t in evt) for evt in data['cluster_Q']]

_, times_python = benchmark("Python nested loop", python_nested_sum)

print(f"\nDSL speedup: {np.mean(times_python)/np.mean(times_dsl):.2f}x")

## Summary

In [ ]:
print("=" * 50)
print("08c_debug_2d.ipynb - L3 2D Array Operations")
print("=" * 50)
print("\n✓ Section 1: 2D column slicing - PASSED")
print("✓ Section 2: 2D row slicing - PASSED")
print("✓ Section 3: 2D combined slicing - PASSED")
print("✓ Section 4: 2D reductions - PASSED")
print("\n" + "=" * 50)
print("ALL TESTS PASSED")
print("=" * 50)